In [1]:
# import scrape_dataset
# scrape_dataset.main()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile, os

with zipfile.ZipFile("/content/drive/MyDrive/dataset2.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")

print(os.listdir("/content/dataset2"))

['aluminium_foil', 'gutka_packet', 'notebook_waste', 'ceramic_broken', 'cigarette_butt', 'polythene_bag', 'metal_can', 'fabric_clothes', 'soiled_tissue', 'plastic_bottle', 'cardboard_box', 'sanitary_napkin', 'glass_bottle', 'broken_glass', 'newspaper_paper', 'chips_packet', 'thermocol', 'tetra_pak', 'diaper', 'steel_utensil']


In [4]:
import tensorflow as tf
import pathlib

data_dir = pathlib.Path("/content/dataset2")
corrupted = []

for f in data_dir.rglob("*"):
    if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}:
        try:
            img = tf.io.read_file(str(f))
            tf.image.decode_image(img)
        except Exception as e:
            corrupted.append(f)
            print(f"Corrupted: {f}")

print(f"\nTotal corrupted: {len(corrupted)}")

Corrupted: /content/dataset2/gutka_packet/000223.jpg
Corrupted: /content/dataset2/gutka_packet/000213.jpg

Total corrupted: 2


In [5]:
for f in corrupted:
    f.unlink()
print("Done!")

Done!


In [6]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

In [7]:
DATASET_PATH = "/content/dataset2"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)
print("Total:", len(class_names))

Found 5064 files belonging to 20 classes.
Using 3545 files for training.
Found 5064 files belonging to 20 classes.
Using 1519 files for validation.
Classes: ['aluminium_foil', 'broken_glass', 'cardboard_box', 'ceramic_broken', 'chips_packet', 'cigarette_butt', 'diaper', 'fabric_clothes', 'glass_bottle', 'gutka_packet', 'metal_can', 'newspaper_paper', 'notebook_waste', 'plastic_bottle', 'polythene_bag', 'sanitary_napkin', 'soiled_tissue', 'steel_utensil', 'tetra_pak', 'thermocol']
Total: 20


In [8]:
total_batches = tf.data.experimental.cardinality(val_test_ds).numpy()
val_size = total_batches // 2

val_ds  = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print(f"Train batches : {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Val batches   : {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Test batches  : {tf.data.experimental.cardinality(test_ds).numpy()}")

Train batches : 111
Val batches   : 24
Test batches  : 24


In [9]:
NUM_CLASSES = len(class_names)

In [10]:
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.15),
    RandomZoom(0.1),
    RandomBrightness(0.1),
    RandomContrast(0.1),
])

train_ds_augmented = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

print("Augmentation ready ✅")

Augmentation ready ✅


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ( Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout ,BatchNormalization , Rescaling,
    RandomFlip, RandomRotation, RandomZoom, RandomBrightness, RandomContrast )

In [12]:
NUM_CLASSES = 20

model = Sequential()
model.add(Input(shape=(224, 224, 3)))
model.add(Rescaling(1./255))

# Block 1
model.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D(2,2))

# Block 2
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Block 3
model.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D(2,2))

# Block 4
model.add(Conv2D(256, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Classifier Head
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(NUM_CLASSES, activation='softmax'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    12,845,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         5,140 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,240,788 (50.51 MB)

 Trainable params: 13,239,828 (50.51 MB)

 Non-trainable params: 960 (3.75 KB)

In [13]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [14]:
history_0 = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=20
)

Epoch 1/20


KeyboardInterrupt: 

In [15]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

transfer_model = tf.keras.Sequential()
transfer_model.add(tf.keras.layers.Rescaling(1./127.5, offset=-1, input_shape=(224, 224, 3)))
transfer_model.add(base_model)
transfer_model.add(tf.keras.layers.GlobalAveragePooling2D())
transfer_model.add(tf.keras.layers.Dense(128, activation='relu'))
transfer_model.add(tf.keras.layers.Dropout(0.4))
transfer_model.add(tf.keras.layers.Dense(64, activation='relu'))
transfer_model.add(tf.keras.layers.Dense(20, activation='softmax'))

transfer_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 20)             │         1,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,431,508 (9.28 MB)

 Trainable params: 173,524 (677.83 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [16]:
transfer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
history = transfer_model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=20
)

Epoch 1/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 77s 522ms/step - accuracy: 0.2516 - loss: 2.5101 - val_accuracy: 0.5234 - val_loss: 1.7186
Epoch 2/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 37s 332ms/step - accuracy: 0.4762 - loss: 1.7615 - val_accuracy: 0.6003 - val_loss: 1.3375
Epoch 3/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 34s 306ms/step - accuracy: 0.5529 - loss: 1.5199 - val_accuracy: 0.6133 - val_loss: 1.2762
Epoch 4/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 35s 314ms/step - accuracy: 0.5915 - loss: 1.3711 - val_accuracy: 0.6289 - val_loss: 1.2351
Epoch 5/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 44s 394ms/step - accuracy: 0.6169 - loss: 1.2820 - val_accuracy: 0.6667 - val_loss: 1.1741
Epoch 6/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 34s 305ms/step - accuracy: 0.6358 - loss: 1.1982 - val_accuracy: 0.6602 - val_loss: 1.1445
Epoch 7/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 34s 307ms/step - accuracy: 0.6581 - loss: 1.1247 - val_accuracy: 0.6667 - val_loss: 1.1012
Epoch 8/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 41s 309ms/step - accuracy: 0.6683 - loss: 1

In [21]:
# Fine Tuning our model
for layer in base_model.layers[-2:]:
    layer.trainable = True

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25
)

Epoch 1/25
111/111 ━━━━━━━━━━━━━━━━━━━━ 54s 394ms/step - accuracy: 0.7882 - loss: 0.6678 - val_accuracy: 0.7214 - val_loss: 0.9791
Epoch 2/25
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.7817 - loss: 0.6944 - val_accuracy: 0.7201 - val_loss: 0.9766
Epoch 3/25
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7842 - loss: 0.6681 - val_accuracy: 0.7214 - val_loss: 0.9740
Epoch 4/25
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.7921 - loss: 0.6621 - val_accuracy: 0.7240 - val_loss: 0.9731
Epoch 5/25
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.7929 - loss: 0.6626 - val_accuracy: 0.7266 - val_loss: 0.9702
Epoch 6/25
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.7941 - loss: 0.6526 - val_accuracy: 0.7240 - val_loss: 0.9678
Epoch 7/25
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7952 - loss: 0.6514 - val_accuracy: 0.7214 - val_loss: 0.9671
Epoch 8/25
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.7913 - loss: 0.6553 - val_ac

In [22]:
transfer_model.save("waste_classifier.keras")

In [24]:
test_loss, test_accuracy = transfer_model.evaluate(test_ds)

print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy*100:.2f}%")

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.6957 - loss: 0.9850

Test Loss     : 0.9850
Test Accuracy : 69.57%
